In [1]:
import databento as db
import dotenv
import os
import pandas as pd
import numpy as np

dotenv.load_dotenv()

client = db.Historical(os.getenv("DATABENTO_API_KEY"))

data = client.timeseries.get_range(
    dataset="GLBX.MDP3",
    symbols=[f"CL.v.0", "ES.v.0"],
    stype_in="continuous",
    schema="ohlcv-1m",
    start="2025-01-02",
    end="2025-02-01",
)

df = data.to_df()

ohlcv_agg = {
    "open": "first",
    "high": "max",
    "low": "min",
    "close": "last",
    "volume": "sum",
}

symbols = {"ES.v.0": "es", "CL.v.0": "cl"}
resampled = {}
for sym, prefix in symbols.items():
    sub = df[df["symbol"] == sym][["open", "high", "low", "close", "volume"]]
    sub = sub.resample("5min").agg(ohlcv_agg)
    sub.columns = [f"{prefix}_{c}" for c in sub.columns]
    resampled[prefix] = sub

df = resampled["es"].join(resampled["cl"], how="outer")
df = df.dropna()
print(df)
df.to_csv(f"data/multi-asset.csv")


                           es_open  es_high   es_low  es_close  es_volume  \
ts_event                                                                    
2025-01-02 00:00:00+00:00  5939.25  5939.50  5927.50   5927.50       2222   
2025-01-02 00:05:00+00:00  5927.50  5932.25  5927.00   5931.25       1780   
2025-01-02 00:10:00+00:00  5931.25  5931.25  5925.50   5928.00       1309   
2025-01-02 00:15:00+00:00  5928.00  5929.50  5910.75   5915.50       5191   
2025-01-02 00:20:00+00:00  5915.75  5917.50  5914.00   5917.50       1217   
...                            ...      ...      ...       ...        ...   
2025-01-31 21:35:00+00:00  6061.25  6063.25  6061.00   6062.25       1350   
2025-01-31 21:40:00+00:00  6062.25  6063.00  6061.50   6062.75       1325   
2025-01-31 21:45:00+00:00  6062.50  6065.25  6062.25   6065.00       1373   
2025-01-31 21:50:00+00:00  6065.00  6067.25  6062.25   6062.75       2616   
2025-01-31 21:55:00+00:00  6062.75  6066.25  6061.25   6065.50       2558   